# Amazon Review Project Checker

This notebook is for checking:
- training / validation / test datasets
- main scored results
- new dataset scored results
- saved metric files
- suspicious and non-suspicious examples

It also includes functions to randomly print example rows in a cleaner format.


In [1]:
import os
import sys
import json
import textwrap
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("Python executable:", sys.executable)
print("Current working directory:", os.getcwd())


Python executable: C:\Users\linnu\anaconda3\python.exe
Current working directory: C:\Users\linnu\Downloads\CSE 404\project


In [2]:
# Project paths
PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "dataset"

TRAIN_PATH = DATASET_DIR / "amazon_reviews_train.csv"
VAL_PATH = DATASET_DIR / "amazon_reviews_val.csv"
TEST_PATH = DATASET_DIR / "amazon_reviews_test.csv"
READY_PATH = DATASET_DIR / "amazon_reviews_ready.csv"
CLEANED_PATH = DATASET_DIR / "amazon_reviews_cleaned.csv"

MAIN_FLAGGED_PATH = DATASET_DIR / "amazon_reviews_flagged_v2.csv"
MAIN_SUMMARY_PATH = DATASET_DIR / "suspicious_review_summary_v2.csv"

NEW_READY_PATH = DATASET_DIR / "new_amazon_test_ready.csv"
NEW_FLAGGED_PATH = DATASET_DIR / "new_amazon_test_with_credibility.csv"
NEW_SUMMARY_PATH = DATASET_DIR / "new_amazon_test_credibility_summary.csv"

EXAMPLES_DIR = DATASET_DIR / "credibility_examples"
SUSPICIOUS_EXAMPLES_PATH = EXAMPLES_DIR / "suspicious_examples.csv"
NOT_SUSPICIOUS_EXAMPLES_PATH = EXAMPLES_DIR / "not_suspicious_examples.csv"

ROBERTA_RESULTS_PATH = DATASET_DIR / "best_single_model_roberta_fast_results.csv"
ROBERTA_REPORT_PATH = DATASET_DIR / "best_single_model_roberta_fast_classification_report.csv"
ROBERTA_CM_PATH = DATASET_DIR / "best_single_model_roberta_fast_confusion_matrix.csv"
ROBERTA_LOG_PATH = DATASET_DIR / "best_single_model_roberta_fast_training_log.csv"

MLP_RESULTS_PATH = DATASET_DIR / "best_single_model_nn_results.csv"
MLP_REPORT_PATH = DATASET_DIR / "best_single_model_nn_classification_report.csv"
MLP_CM_PATH = DATASET_DIR / "best_single_model_nn_confusion_matrix.csv"

BEST_V3_RESULTS_PATH = DATASET_DIR / "best_single_model_v3_results.csv"
BEST_V3_REPORT_PATH = DATASET_DIR / "best_single_model_v3_classification_report.csv"
BEST_V3_CM_PATH = DATASET_DIR / "best_single_model_v3_confusion_matrix.csv"

COMBINED_MODEL_PATH = DATASET_DIR / "combined_review_credibility_model.joblib"

all_paths = {
    "TRAIN_PATH": TRAIN_PATH,
    "VAL_PATH": VAL_PATH,
    "TEST_PATH": TEST_PATH,
    "READY_PATH": READY_PATH,
    "CLEANED_PATH": CLEANED_PATH,
    "MAIN_FLAGGED_PATH": MAIN_FLAGGED_PATH,
    "MAIN_SUMMARY_PATH": MAIN_SUMMARY_PATH,
    "NEW_READY_PATH": NEW_READY_PATH,
    "NEW_FLAGGED_PATH": NEW_FLAGGED_PATH,
    "NEW_SUMMARY_PATH": NEW_SUMMARY_PATH,
    "SUSPICIOUS_EXAMPLES_PATH": SUSPICIOUS_EXAMPLES_PATH,
    "NOT_SUSPICIOUS_EXAMPLES_PATH": NOT_SUSPICIOUS_EXAMPLES_PATH,
    "COMBINED_MODEL_PATH": COMBINED_MODEL_PATH,
}

status_df = pd.DataFrame({
    "name": list(all_paths.keys()),
    "path": [str(p) for p in all_paths.values()],
    "exists": [p.exists() for p in all_paths.values()],
})

display(status_df)


,name,path,exists
0,TRAIN_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\amazon_reviews_train.csv,True
1,VAL_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\amazon_reviews_val.csv,True
2,TEST_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\amazon_reviews_test.csv,True
3,READY_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\amazon_reviews_ready.csv,True
4,CLEANED_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\amazon_reviews_cleaned.csv,True
5,MAIN_FLAGGED_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\amazon_reviews_flagged_v2.csv,True
6,MAIN_SUMMARY_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\suspicious_review_summary_v2.csv,True
7,NEW_READY_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\new_amazon_test_ready.csv,True
8,NEW_FLAGGED_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\new_amazon_test_with_credibility.csv,True
9,NEW_SUMMARY_PATH,C:\Users\linnu\Downloads\CSE 404\project\dataset\new_amazon_test_credibility_summary.csv,True


In [3]:
def safe_read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    return pd.read_csv(path, low_memory=False, **kwargs)


def dataset_overview(df, name="dataset"):
    if df is None:
        print(f"{name}: not loaded")
        return

    print(f"===== {name} =====")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if "sentiment" in df.columns:
        print("\nSentiment distribution:")
        display(
            df["sentiment"]
            .value_counts(dropna=False)
            .rename_axis("sentiment")
            .reset_index(name="count")
        )

    if "category" in df.columns:
        print("\nCategory distribution:")
        display(
            df["category"]
            .value_counts(dropna=False)
            .rename_axis("category")
            .reset_index(name="count")
        )

    display(df.head(3))


def quick_result_summary(df, name="result"):
    if df is None:
        print(f"{name}: not loaded")
        return

    print(f"===== {name} =====")
    print("Shape:", df.shape)

    if "suspicious_label" in df.columns:
        print("\nSuspicious label distribution:")
        display(
            df["suspicious_label"]
            .value_counts(dropna=False, normalize=True)
            .mul(100)
            .round(2)
            .rename_axis("suspicious_label")
            .reset_index(name="percent")
        )

    if "low_credibility_label" in df.columns:
        print("\nLow credibility label distribution:")
        display(
            df["low_credibility_label"]
            .value_counts(dropna=False, normalize=True)
            .mul(100)
            .round(2)
            .rename_axis("low_credibility_label")
            .reset_index(name="percent")
        )

    if "predicted_sentiment" in df.columns:
        print("\nPredicted sentiment distribution:")
        display(
            df["predicted_sentiment"]
            .value_counts(dropna=False, normalize=True)
            .mul(100)
            .round(2)
            .rename_axis("predicted_sentiment")
            .reset_index(name="percent")
        )

    if "low_credibility_score" in df.columns:
        print("\nScore stats:")
        display(df["low_credibility_score"].describe().to_frame("low_credibility_score"))


def display_random_rows(df, n=10, random_state=42, suspicious_only=None, text_width=70):
    if df is None or len(df) == 0:
        print("No rows to display.")
        return

    work = df.copy()

    if suspicious_only is True and "suspicious_label" in work.columns:
        work = work[work["suspicious_label"] == "suspicious"].copy()
    elif suspicious_only is False and "suspicious_label" in work.columns:
        work = work[work["suspicious_label"] != "suspicious"].copy()

    if len(work) == 0:
        print("No matching rows found.")
        return

    preferred_cols = [
        "category",
        "text",
        "sentiment",
        "predicted_sentiment",
        "low_credibility_score",
        "low_credibility_label",
        "suspicious_label",
        "suspicious_reasons",
    ]
    show_cols = [c for c in preferred_cols if c in work.columns]

    sample = work.sample(n=min(n, len(work)), random_state=random_state)[show_cols].copy()

    if "text" in sample.columns:
        sample["text"] = sample["text"].astype(str).apply(
            lambda x: "\n".join(textwrap.wrap(x, width=text_width))
        )

    if "suspicious_reasons" in sample.columns:
        sample["suspicious_reasons"] = sample["suspicious_reasons"].astype(str).apply(
            lambda x: "\n".join(textwrap.wrap(x, width=40))
        )

    if "low_credibility_score" in sample.columns:
        sample["low_credibility_score"] = sample["low_credibility_score"].round(3)

    sample = sample.reset_index(drop=True)
    sample.insert(0, "row", range(1, len(sample) + 1))
    display(sample)


def random_print_10(df, random_state=42):
    display_random_rows(df, n=10, random_state=random_state)


## 1. Check core datasets

In [4]:
train_df = safe_read_csv(TRAIN_PATH)
val_df = safe_read_csv(VAL_PATH)
test_df = safe_read_csv(TEST_PATH)
ready_df = safe_read_csv(READY_PATH)
cleaned_df = safe_read_csv(CLEANED_PATH, nrows=1000)  # cleaned file is very large
new_ready_df = safe_read_csv(NEW_READY_PATH)

dataset_overview(train_df, "train_df")
dataset_overview(val_df, "val_df")
dataset_overview(test_df, "test_df")
dataset_overview(ready_df, "ready_df")
dataset_overview(cleaned_df, "cleaned_df_sample")
dataset_overview(new_ready_df, "new_ready_df")


===== train_df =====
Shape: (84000, 9)
Columns: ['category', 'asin', 'overall', 'sentiment', 'verified', 'vote', 'unixReviewTime', 'text', 'clean_text']

Sentiment distribution:


,sentiment,count
0,positive,70732
1,negative,7176
2,neutral,6092



Category distribution:


,category,count
0,Cell Phones and Accessories,28000
1,Sports and Outdoors,28000
2,Industrial and Scientific,28000


,category,asin,overall,sentiment,verified,vote,unixReviewTime,text,clean_text
0,Cell Phones and Accessories,B00LOVW1AQ,1.0,negative,True,4,1439164800,Do not buy this phone! I have been using this phone for 3 days and I will be returning it. The home screen pages tak...,do not buy this phone i have been using this phone for 3 days and i will be returning it the home screen pages take ...
1,Cell Phones and Accessories,B004HF8MP4,1.0,negative,True,3,1319587200,"Phone was junk I bought this phone and from day one it had problems. First, the battery would go dead really really ...",phone was junk i bought this phone and from day one it had problems first the battery would go dead really really fa...
2,Cell Phones and Accessories,B0128FAJ32,4.0,positive,True,NaN,1505606400,Honest review- will change rating if changes occur overtime This portable charger works like any charger should. Giv...,honest review will change rating if changes occur overtime this portable charger works like any charger should given...


...

===== new_ready_df =====
Shape: (50000, 9)
Columns: ['category', 'asin', 'overall', 'sentiment', 'verified', 'vote', 'unixReviewTime', 'text', 'clean_text']

Sentiment distribution:


,sentiment,count
0,positive,44310
1,neutral,2911
2,negative,2779



Category distribution:


,category,count
0,Arts Crafts and Sewing,50000


,category,asin,overall,sentiment,verified,vote,unixReviewTime,text,clean_text
0,Arts Crafts and Sewing,B008SOZK2I,5.0,positive,True,NaN,1444176000,Very good deal. A very good buy for a jewlery maker(me),very good deal a very good buy for a jewlery maker me
1,Arts Crafts and Sewing,B0109326CM,5.0,positive,True,NaN,1484352000,Five Stars. good,five stars good
2,Arts Crafts and Sewing,B0012SG11Q,4.0,positive,True,NaN,1454371200,Four Stars. ok,four stars ok


## 2. Check saved model metric files

In [5]:
metric_files = [
    ("RoBERTa fast results", ROBERTA_RESULTS_PATH),
    ("RoBERTa fast report", ROBERTA_REPORT_PATH),
    ("RoBERTa fast confusion matrix", ROBERTA_CM_PATH),
    ("RoBERTa fast training log", ROBERTA_LOG_PATH),
    ("MLP results", MLP_RESULTS_PATH),
    ("MLP report", MLP_REPORT_PATH),
    ("MLP confusion matrix", MLP_CM_PATH),
    ("Best traditional results", BEST_V3_RESULTS_PATH),
    ("Best traditional report", BEST_V3_REPORT_PATH),
    ("Best traditional confusion matrix", BEST_V3_CM_PATH),
]

for name, path in metric_files:
    print(f"\n===== {name} =====")
    if Path(path).exists():
        display(pd.read_csv(path))
    else:
        print("Missing file:", path)



===== RoBERTa fast results =====


,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,loss
0,0.814667,0.812417,0.814667,0.812662,0.812417,0.814667,0.812662,0.515848



===== RoBERTa fast report =====


,Unnamed: 0,precision,recall,f1-score,support
0,negative,0.820312,0.840000,0.830040,500.000000
1,neutral,0.767033,0.698000,0.730890,500.000000
2,positive,0.849906,0.906000,0.877057,500.000000
3,accuracy,0.814667,0.814667,0.814667,0.814667
4,macro avg,0.812417,0.814667,0.812662,1500.000000
5,weighted avg,0.812417,0.814667,0.812662,1500.000000



===== RoBERTa fast confusion matrix =====


...

,split,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,config_json
0,validation,0.919333,0.799596,0.704332,0.740658,0.910197,0.919333,0.911969,"{""word_max_features"": 70000, ""char_max_features"": 50000, ""word_ngram_range"": [1, 2], ""char_ngram_range"": [2, 5], ""wo..."
1,test,0.917556,0.792792,0.698067,0.735188,0.907910,0.917556,0.910145,"{""word_max_features"": 70000, ""char_max_features"": 50000, ""word_ngram_range"": [1, 2], ""char_ngram_range"": [2, 5], ""wo..."



===== Best traditional report =====


,label,precision,recall,f1-score,support
0,negative,0.799419,0.714750,0.754717,1539.000000
1,neutral,0.636700,0.396472,0.488658,1304.000000
2,positive,0.942259,0.982978,0.962188,15157.000000
3,accuracy,0.917556,0.917556,0.917556,0.917556
4,macro avg,0.792792,0.698067,0.735188,18000.000000
5,weighted avg,0.907910,0.917556,0.910145,18000.000000



===== Best traditional confusion matrix =====


,Unnamed: 0,negative,neutral,positive
0,negative,1100,142,297
1,neutral,171,517,616
2,positive,105,153,14899


## 3. Check main dataset credibility results

In [6]:
main_flagged_df = safe_read_csv(MAIN_FLAGGED_PATH)
main_summary_df = safe_read_csv(MAIN_SUMMARY_PATH)

quick_result_summary(main_flagged_df, "main_flagged_df")

print("\nMain summary file:")
display(main_summary_df)


===== main_flagged_df =====
Shape: (120000, 25)

Suspicious label distribution:


,suspicious_label,percent
0,not_suspicious,97.92
1,suspicious,2.08



Low credibility label distribution:


,low_credibility_label,percent
0,low,91.22
1,medium,7.54
2,high,1.24



Predicted sentiment distribution:


,predicted_sentiment,percent
0,positive,77.96
1,neutral,12.56
2,negative,9.48



Score stats:


,low_credibility_score
count,120000.000000
mean,0.109228
std,0.123338
min,0.000000
25%,0.000000
50%,0.100000
75%,0.180000
max,0.740000



Main summary file:


,group_name,row_count,suspicious_rate,high_risk_rate,avg_score
0,ALL,120000,0.02082,0.01236,0.10923
1,Cell Phones and Accessories,40000,0.02342,0.01622,0.11476
2,Industrial and Scientific,40000,0.03012,0.01605,0.11744
3,Sports and Outdoors,40000,0.00890,0.00480,0.09548


In [13]:
print("=== 10 Random Main Result Rows ===")
display_random_rows(main_flagged_df, n=10, random_state=42)

print("=== 10 Random Suspicious Rows From Main Result ===")
display_random_rows(main_flagged_df, n=10, random_state=42, suspicious_only=True)

print("=== 10 Random Not Suspicious Rows From Main Result ===")
display_random_rows(main_flagged_df, n=10, random_state=42, suspicious_only=False)


=== 10 Random Main Result Rows ===


,row,category,text,sentiment,predicted_sentiment,low_credibility_score,low_credibility_label,suspicious_label,suspicious_reasons
0,1,Sports and Outdoors,Comfy Liners These glove liners make do make a difference and will\nkeep your hands warm. They are thin and fit insi...,positive,positive,0.10,low,not_suspicious,high_token_repetition
1,2,Sports and Outdoors,Four Stars seems well made but I have not used it yet.,positive,positive,0.00,low,not_suspicious,none
2,3,Sports and Outdoors,Poor Doesn't work worth a damn.,negative,negative,0.00,low,not_suspicious,none
3,4,Cell Phones and Accessories,Great first time use VR This is my first VR and it is great. It give\nme true feeling. A lot cheaper than samsung g...,positive,positive,0.00,low,not_suspicious,none
4,5,Cell Phones and Accessories,Three Stars ok,neutral,neutral,0.12,low,not_suspicious,very_short_text
5,6,Industrial and Scientific,"A True RMS AC meter that takes the guess work out of setup and using.\n<div id=""video-block-R1HJ9SMR25D70Q"" class=""a...",positive,positive,0.18,low,not_suspicious,not_verified_purchase;\nhigh_token_repetition
6,7,Cell Phones and Accessories,"sturdy construction.. like a seat belt. the metal clip was the only\npart I had a problem with. So, I did what I alw...",positive,positive,0.10,low,not_suspicious,high_token_repetition
7,8,Cell Phones and Accessories,"Some troubles when installing, but it's not bad Installation was not\nso easy. It ended up with a bit of air bubbles...",positive,neutral,0.00,low,not_suspicious,none
8,9,Cell Phones and Accessories,I think it was a good value for the price This case fit my son'r\nGalaxy S3 perfectly. I think it was a good value f...,positive,positive,0.10,low,not_suspicious,high_token_repetition
9,10,Industrial and Scientific,very nice orange PLA filament very nice orange PLA filament. it prints\nnicely and the finish looks great. I haven't...,positive,positive,0.00,low,not_suspicious,none


=== 10 Random Suspicious Rows From Main Result ===


,row,category,text,sentiment,predicted_sentiment,low_credibility_score,low_credibility_label,suspicious_label,suspicious_reasons
0,1,Industrial and Scientific,Five Stars great,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
1,2,Industrial and Scientific,Five Stars great,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
2,3,Cell Phones and Accessories,Five Stars Love it,positive,positive,0.52,medium,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text
3,4,Industrial and Scientific,Five Stars Good,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
4,5,Sports and Outdoors,Five Stars Great,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
5,6,Cell Phones and Accessories,Five Stars Good,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
6,7,Industrial and Scientific,Five Stars works great,positive,positive,0.52,medium,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text
7,8,Cell Phones and Accessories,Four Stars good,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
8,9,Cell Phones and Accessories,Four Stars Nice,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
9,10,Cell Phones and Accessories,Five Stars Excellent!!!...,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text


=== 10 Random Not Suspicious Rows From Main Result ===


,row,category,text,sentiment,predicted_sentiment,low_credibility_score,low_credibility_label,suspicious_label,suspicious_reasons
0,1,Sports and Outdoors,Works as intended with little set up Pefect adapters to make a Sawyer\nfiltration fit camelback. Allows easy fill o...,positive,positive,0.00,low,not_suspicious,none
1,2,Cell Phones and Accessories,Horrible service I received one today with empty bag Nothing's in it\nIt should be very hard to open it,negative,negative,0.00,low,not_suspicious,none
2,3,Cell Phones and Accessories,"Revised Review I originally had complaints about the product as\ninstallation was rather difficult, but Flexion made...",positive,positive,0.10,low,not_suspicious,high_token_repetition
3,4,Sports and Outdoors,A great little gift I gave this knife as a gift to a family member who\nis an artist. She makes great use of it in h...,positive,positive,0.00,low,not_suspicious,none
4,5,Cell Phones and Accessories,"Overall good. The item arrived so fast and as expected, the item seems\nto be nice and neat. It's been 2 days now us...",positive,positive,0.00,low,not_suspicious,none
5,6,Industrial and Scientific,Bright Tape That Holds Well And Is Easy To Cut. I have used this tape\nto seal boxes as well as to mark spots on my ...,positive,positive,0.18,low,not_suspicious,not_verified_purchase;\nhigh_token_repetition
6,7,Cell Phones and Accessories,"Great product! Perfect case, fits well and is great for protecting\nyour phone from falls.",positive,positive,0.00,low,not_suspicious,none
7,8,Sports and Outdoors,If you use the ams bowfishing reel its too long ... If you use the ams\nbowfishing reel its too long so I took a gri...,negative,neutral,0.10,low,not_suspicious,high_token_repetition
8,9,Cell Phones and Accessories,Five Stars ok,positive,positive,0.34,medium,not_suspicious,very_short_generic_text; very_short_text
9,10,Cell Phones and Accessories,"Decent, side buttons harder to press It's an OK minimal case. The\ncutouts make the side buttons (volume up/down an...",neutral,neutral,0.00,low,not_suspicious,none


## 4. Check new dataset credibility results

In [14]:
new_flagged_df = safe_read_csv(NEW_FLAGGED_PATH)
new_summary_df = safe_read_csv(NEW_SUMMARY_PATH)

quick_result_summary(new_flagged_df, "new_flagged_df")

print("\nNew dataset summary file:")
display(new_summary_df)


===== new_flagged_df =====
Shape: (50000, 25)

Suspicious label distribution:


,suspicious_label,percent
0,not_suspicious,96.39
1,suspicious,3.61



Low credibility label distribution:


,low_credibility_label,percent
0,low,89.76
1,medium,7.96
2,high,2.28



Predicted sentiment distribution:


,predicted_sentiment,percent
0,positive,84.38
1,neutral,9.29
2,negative,6.33



Score stats:


,low_credibility_score
count,50000.000000
mean,0.115647
std,0.139789
min,0.000000
25%,0.000000
50%,0.100000
75%,0.220000
max,0.740000



New dataset summary file:


,group_name,row_count,suspicious_rate,high_risk_rate,avg_score
0,ALL,50000,0.03608,0.0228,0.11565
1,Arts Crafts and Sewing,50000,0.03608,0.0228,0.11565


In [15]:
print("=== 10 Random New Dataset Result Rows ===")
display_random_rows(new_flagged_df, n=10, random_state=42)

print("=== 10 Random Suspicious Rows From New Dataset ===")
display_random_rows(new_flagged_df, n=10, random_state=42, suspicious_only=True)

print("=== 10 Random Not Suspicious Rows From New Dataset ===")
display_random_rows(new_flagged_df, n=10, random_state=42, suspicious_only=False)


=== 10 Random New Dataset Result Rows ===


,row,category,text,sentiment,predicted_sentiment,low_credibility_score,low_credibility_label,suspicious_label,suspicious_reasons
0,1,Arts Crafts and Sewing,Five Stars. Made a great backing for a wall hanging,positive,positive,0.0,low,not_suspicious,none
1,2,Arts Crafts and Sewing,Works but handle is too short. It's a nice price and it works well. As\nother reviewers have mentioned the handle is...,positive,neutral,0.1,low,not_suspicious,high_token_repetition
2,3,Arts Crafts and Sewing,Two Stars. It's ok,negative,negative,0.0,low,not_suspicious,none
3,4,Arts Crafts and Sewing,I have purchased several of these as they are one .... I have\npurchased several of these as they are one of the mai...,positive,positive,0.1,low,not_suspicious,high_token_repetition
4,5,Arts Crafts and Sewing,"great nonstick pressing sheets. These sheets are awesome! I use a lot\nof fusible tape/webbing, and have had no iss...",positive,positive,0.0,low,not_suspicious,none
5,6,Arts Crafts and Sewing,Five Stars. received my order quickly great product and color choices.\nI use this vinyl with my Cricut Air.,positive,positive,0.0,low,not_suspicious,none
6,7,Arts Crafts and Sewing,Pin Me. cannot live without,positive,positive,0.0,low,not_suspicious,none
7,8,Arts Crafts and Sewing,"Stamp pads for kids. Great for my grandkids stamping projects, they\nlove them. Great Price as well.",positive,positive,0.0,low,not_suspicious,none
8,9,Arts Crafts and Sewing,Three Stars. Does make it easier to layout my ideas.,neutral,neutral,0.0,low,not_suspicious,none
9,10,Arts Crafts and Sewing,Best needles. These are the best needles I have ever used. They are\nsharp and very strong. I find that I don't have...,positive,positive,0.1,low,not_suspicious,high_token_repetition


=== 10 Random Suspicious Rows From New Dataset ===


,row,category,text,sentiment,predicted_sentiment,low_credibility_score,low_credibility_label,suspicious_label,suspicious_reasons
0,1,Arts Crafts and Sewing,Five Stars. excellent,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
1,2,Arts Crafts and Sewing,Five Stars. Great product,positive,positive,0.52,medium,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text
2,3,Arts Crafts and Sewing,Five Stars. Good,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
3,4,Arts Crafts and Sewing,Five Stars. Love it :},positive,positive,0.52,medium,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text
4,5,Arts Crafts and Sewing,Five Stars. Great!,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
5,6,Arts Crafts and Sewing,Five Stars. Good,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
6,7,Arts Crafts and Sewing,Five Stars. Great,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
7,8,Arts Crafts and Sewing,Five Stars. Great,positive,positive,0.64,high,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text; very_short_text
8,9,Arts Crafts and Sewing,Five Stars. love it!!,positive,positive,0.52,medium,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text
9,10,Arts Crafts and Sewing,Five Stars. as expected,positive,positive,0.52,medium,suspicious,duplicate_or_repeated_text;\nvery_short_generic_text


=== 10 Random Not Suspicious Rows From New Dataset ===


,row,category,text,sentiment,predicted_sentiment,low_credibility_score,low_credibility_label,suspicious_label,suspicious_reasons
0,1,Arts Crafts and Sewing,Five Stars. nice clean batting for quilting...not thick.,positive,positive,0.00,low,not_suspicious,none
1,2,Arts Crafts and Sewing,"great handles - no short cuts used in manufacturing these ....\nQuality, smooth hooks , great handles - no short cut...",positive,positive,0.10,low,not_suspicious,high_token_repetition
2,3,Arts Crafts and Sewing,Perfect for non-fancy crafting.. This was perfect for making pom poms\nfor Xmas decorating.,positive,positive,0.00,low,not_suspicious,none
3,4,Arts Crafts and Sewing,Five Stars. VERY HAPPY,positive,positive,0.22,low,not_suspicious,very_short_generic_text
4,5,Arts Crafts and Sewing,Four Stars. Looks and feels durable.,positive,positive,0.22,low,not_suspicious,very_short_generic_text
5,6,Arts Crafts and Sewing,Fun and addictive craft. Fun and interesting craft. And it's a good\ndeal on Amazon. I'm taking off one star becaus...,positive,positive,0.18,low,not_suspicious,not_verified_purchase;\nhigh_token_repetition
6,7,Arts Crafts and Sewing,Convenient and easy to use. Delivered on time and as advertised.\nConvenient and easy to use. The black paint is mor...,positive,positive,0.00,low,not_suspicious,none
7,8,Arts Crafts and Sewing,Not for face painting. It says that the product should not be used\nnear the eye.,negative,negative,0.00,low,not_suspicious,none
8,9,Arts Crafts and Sewing,Tape is alright not the greatest for my uses.. Tape claims to be\npermanent after 24 hours. 72 hours later and the t...,neutral,neutral,0.00,low,not_suspicious,none
9,10,Arts Crafts and Sewing,Make your pressing 100% easier. I am doing some garment sewing and was\nstruggling with keeping the pressing ham in ...,positive,positive,0.00,low,not_suspicious,none


## 5. Check saved example CSVs

In [16]:
suspicious_examples_df = safe_read_csv(SUSPICIOUS_EXAMPLES_PATH)
not_suspicious_examples_df = safe_read_csv(NOT_SUSPICIOUS_EXAMPLES_PATH)

print("Saved suspicious examples:")
display(suspicious_examples_df)

print("Saved not suspicious examples:")
display(not_suspicious_examples_df)


Saved suspicious examples:


,category,asin,overall,sentiment,verified,vote,unixReviewTime,text,clean_text,predicted_sentiment,duplicate_or_repeated_text,word_count,char_count,unique_ratio,repetition_ratio,very_short_text,generic_short_text,high_repetition,verified_bool,unverified_flag,rating_text_mismatch,low_credibility_score,low_credibility_label,suspicious_label,suspicious_reasons
0,Arts Crafts and Sewing,B000XZYLU2,5.0,positive,True,NaN,1430784000,nice. Nice,nice nice,positive,True,2,9,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
1,Arts Crafts and Sewing,B000OVNPXO,5.0,positive,True,NaN,1461456000,Nice. Nice,nice nice,positive,True,2,9,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
2,Arts Crafts and Sewing,B009AG9LQY,5.0,positive,True,NaN,1427846400,good. good,good good,positive,True,2,9,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
3,Arts Crafts and Sewing,B000BROV02,4.0,positive,True,NaN,1427068800,good. Good,good good,positive,True,2,9,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
4,Arts Crafts and Sewing,B0019K3CEQ,5.0,positive,True,NaN,1438214400,Excellent. Excellent :),excellent excellent,positive,True,2,19,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
5,Arts Crafts and Sewing,B000MRR3GU,5.0,positive,True,NaN,1455840000,Excellent. Excellent,excellent excellent,positive,True,2,19,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
6,Arts Crafts and Sewing,B000RAWDJO,5.0,positive,True,NaN,1426982400,great. great,great great,positive,True,2,11,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
7,Arts Crafts and Sewing,B000XAR0DM,5.0,positive,True,NaN,1439078400,Great. Great,great great,positive,True,2,11,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
8,Arts Crafts and Sewing,B00HC2MEKI,4.0,positive,True,NaN,1428019200,Great. Great,great great,positive,True,2,11,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text
9,Arts Crafts and Sewing,B00BNEQ0DI,5.0,positive,True,NaN,1518134400,great. great,great great,positive,True,2,11,0.5,1.0,True,True,True,True,False,False,0.74,high,suspicious,duplicate_or_repeated_text; very_short_generic_text; high_token_repetition; very_short_text


Saved not suspicious examples:


,category,asin,overall,sentiment,verified,vote,unixReviewTime,text,clean_text,predicted_sentiment,duplicate_or_repeated_text,word_count,char_count,unique_ratio,repetition_ratio,very_short_text,generic_short_text,high_repetition,verified_bool,unverified_flag,rating_text_mismatch,low_credibility_score,low_credibility_label,suspicious_label,suspicious_reasons
0,Arts Crafts and Sewing,B000YZ5EV6,5.0,positive,True,NaN,1511913600,Five Stars. nice clean batting for quilting...not thick.,five stars nice clean batting for quilting not thick,positive,False,9,52,1.000000,0.000000,False,False,False,True,False,False,0.00,low,not_suspicious,none
1,Arts Crafts and Sewing,B01E814JHQ,5.0,positive,True,NaN,1480291200,"great handles - no short cuts used in manufacturing these .... Quality, smooth hooks , great handles - no short cuts...",great handles no short cuts used in manufacturing these quality smooth hooks great handles no short cuts used in man...,positive,False,25,155,0.560000,0.840000,False,False,True,True,False,False,0.10,low,not_suspicious,high_token_repetition
2,Arts Crafts and Sewing,B00281K7UW,5.0,positive,True,NaN,1466208000,Perfect for non-fancy crafting.. This was perfect for making pom poms for Xmas decorating.,perfect for non fancy crafting this was perfect for making pom poms for xmas decorating,positive,False,15,87,0.800000,0.333333,False,False,False,True,False,False,0.00,low,not_suspicious,none
3,Arts Crafts and Sewing,B00JCGVV56,5.0,positive,True,2.0,1453939200,Five Stars. VERY HAPPY,five stars very happy,positive,False,4,21,1.000000,0.000000,False,True,False,True,False,False,0.22,low,not_suspicious,very_short_generic_text
4,Arts Crafts and Sewing,B000WOM50W,4.0,positive,True,NaN,1410825600,Four Stars. Looks and feels durable.,four stars looks and feels durable,positive,False,6,34,1.000000,0.000000,False,True,False,True,False,False,0.22,low,not_suspicious,very_short_generic_text
5,Arts Crafts and Sewing,B001O5ULP4,4.0,positive,False,5.0,1404259200,Fun and addictive craft. Fun and interesting craft. And it's a good deal on Amazon.\n\nI'm taking off one star becau...,fun and addictive craft fun and interesting craft and it s a good deal on amazon i m taking off one star because the...,positive,False,124,613,0.661290,0.491935,False,False,True,False,True,False,0.18,low,not_suspicious,not_verified_purchase; high_token_repetition
6,Arts Crafts and Sewing,B00436SEF0,4.0,positive,True,NaN,1425600000,Convenient and easy to use. Delivered on time and as advertised. Convenient and easy to use. The black paint is more...,convenient and easy to use delivered on time and as advertised convenient and easy to use the black paint is more du...,positive,False,30,159,0.800000,0.366667,False,False,False,True,False,False,0.00,low,not_suspicious,none
7,Arts Crafts and Sewing,B00C2RL1L6,1.0,negative,True,NaN,1464825600,Not for face painting. It says that the product should not be used near the eye.,not for face painting it says that the product should not be used near the eye,negative,False,16,78,0.875000,0.250000,False,False,False,True,False,False,0.00,low,not_suspicious,none
8,Arts Crafts and Sewing,B006TMZDNC,3.0,neutral,True,NaN,1480550400,Tape is alright not the greatest for my uses.. Tape claims to be permanent after 24 hours. 72 hours later and the ta...,tape is alright not the greatest for my uses tape claims to be permanent after 24 hours 72 hours later and the tape ...,neutral,False,37,189,0.783784,0.378378,False,False,False,True,False,False,0.00,low,not_suspicious,none
9,Arts Crafts and Sewing,B003KW9PHO,5.0,positive,True,NaN,1393459200,Make your pressing 100% easier. I am doing some garment sewing and was struggling with keeping the pressing ham in p...,make your pressing 100 easier i am doing some garment sewing and was struggling with keeping the pressing ham in pla...,positive,False,40,210,0.925000,0.150000,False,False,False,True,False,False,0.00,low,not_suspicious,none


## 6. Optional: load combined model metadata

In [17]:
try:
    import __main__
    import joblib
    from train_roberta_model import RobertaSentimentPredictor

    __main__.RobertaSentimentPredictor = RobertaSentimentPredictor

    if not COMBINED_MODEL_PATH.exists():
        raise FileNotFoundError(f"Combined model not found: {COMBINED_MODEL_PATH}")

    combined_bundle = joblib.load(COMBINED_MODEL_PATH)

    print("Loaded combined credibility model.")
    print("Model type:", combined_bundle.get("model_type"))
    print("Source sentiment bundle:", combined_bundle.get("source_sentiment_bundle"))
    print("\nCredibility config:")
    print(json.dumps(combined_bundle.get("config", {}), indent=2))
except Exception as e:
    print("Could not load combined model metadata.")
    print(type(e).__name__, "-", e)


Loaded combined credibility model.
Model type: combined_sentiment_plus_credibility_scorer
Source sentiment bundle: dataset\best_single_review_model_roberta_fast.joblib

Credibility config:
{
  "duplicate_weight": 0.3,
  "generic_weight": 0.22,
  "mismatch_weight": 0.22,
  "unverified_weight": 0.08,
  "repetitive_weight": 0.1,
  "very_short_weight": 0.12,
  "medium_threshold": 0.3,
  "high_threshold": 0.6,
  "suspicious_threshold": 0.45
}
